In [ ]:
# Concept vector (CAV) evaluation: sparsity, exact-zero fraction, selectivity,
# overlap (single file) + cross-run stability (directory)
import os
import glob
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment


# ---------- Helpers ----------
def _orient_atoms(C: torch.Tensor) -> torch.Tensor:
    """Ensure rows are atoms (n_atoms x n_features)."""
    if C.ndim != 2:
        raise ValueError(f"Concept matrix must be 2D, got {tuple(C.shape)}")
    return C if C.size(0) <= C.size(1) else C.T.contiguous()


# ---------- Atom-centric metrics (single file) ----------

def compute_exact_zero_fraction(C: torch.Tensor) -> float:
    """Fraction of entries that are *exactly* 0.0. Dense decompositions (SNMF/PCA/SAE
    decoder weights) are almost never exactly zero, so this is usually ~0 and the
    near-zero threshold metric below is the meaningful one for those methods."""
    return (C == 0).float().mean().item()


def compute_atom_sparsity(C: torch.Tensor, threshold: float = 1e-2, relative: bool = True) -> float:
    """
    Fraction of near-zero values per atom, averaged across atoms.

    threshold is RELATIVE (default) to each atom's own max |value| rather than an
    absolute cutoff: concept vector scales vary a lot across decomposition methods
    (PCA vs SNMF vs SAE codes), so an absolute threshold like 0.1 can silently flag
    ~100% of entries as "sparse" regardless of real structure. Pass relative=False
    to use `threshold` as an absolute cutoff instead.
    """
    if relative:
        atom_scale = C.abs().max(dim=1, keepdim=True).values.clamp_min(1e-12)
        near_zero = C.abs() < threshold * atom_scale
    else:
        near_zero = C.abs() < threshold
    per_atom_sparsity = near_zero.float().mean(dim=1)
    return per_atom_sparsity.mean().item()


def compute_atom_selectivity(C: torch.Tensor) -> float:
    """Hoyer sparsity (0..1) averaged over atoms; higher = more concentrated atoms."""
    eps = 1e-12
    n = C.size(1)
    sqrt_n = torch.sqrt(torch.tensor(float(n), device=C.device))
    l1 = C.abs().sum(dim=1)
    l2 = torch.linalg.norm(C, dim=1) + eps
    s = (sqrt_n - (l1 / l2)) / (sqrt_n - 1.0 + eps)
    return s.clamp_(0.0, 1.0).mean().item()


def compute_concept_overlap_tensor(C: torch.Tensor, normalize_rows: bool = True, threshold: float = 1e-3) -> float:
    """
    Mean off-diagonal cosine similarity between atoms (rows).
    Lower overlap = more disentangled concepts.

    normalize_rows defaults to True because this metric is documented/labeled as
    cosine similarity: without row normalization it silently returns a raw dot
    product instead whenever atoms aren't already unit-norm.
    """
    if C.ndim != 2:
        raise ValueError(f"Expected 2D concept matrix, got {tuple(C.shape)}")

    C_thr = C.clone()
    C_thr[C_thr.abs() < threshold] = 0.0

    if normalize_rows:
        Cn = C_thr / (C_thr.norm(dim=1, keepdim=True) + 1e-12)
    else:
        Cn = C_thr

    sim = Cn @ Cn.T  # shape: (K, K)
    n = sim.size(0)
    if n <= 1:
        return 0.0

    mask = ~torch.eye(n, dtype=torch.bool, device=C.device)
    return sim[mask].mean().item()


# ---------- Cross-run stability (directory) ----------
def _mean_matched_cosine(V1: np.ndarray, V2: np.ndarray) -> float:
    """Mean matched cosine similarity between two atom banks (rows=atoms).
    Uses the full rectangular similarity matrix so no atoms are discarded by
    truncation when the two banks have different atom counts."""
    V1n = V1 / (np.linalg.norm(V1, axis=1, keepdims=True) + 1e-12)
    V2n = V2 / (np.linalg.norm(V2, axis=1, keepdims=True) + 1e-12)
    S = V1n @ V2n.T  # (n1, n2), rectangular is fine for linear_sum_assignment
    row_ind, col_ind = linear_sum_assignment(-S)
    return float(S[row_ind, col_ind].mean())


def compute_cross_run_stability(base_dir: str, pattern: str) -> float:
    """
    Mean matched cosine similarity between concept banks found under base_dir
    (higher = more stable/reproducible across runs, standard convention).

    IMPORTANT: only point base_dir at runs that are genuinely repeats of the SAME
    pipeline config on the SAME data (e.g. different seeds). Globbing broadly across
    unrelated datasets/experiments will average together comparisons that have
    nothing to do with reproducibility and produce a meaningless number.
    """
    paths = sorted(glob.glob(os.path.join(base_dir, pattern), recursive=True))
    if len(paths) < 2:
        print(f"[stability] found {len(paths)} matching file(s) under {base_dir!r} "
              f"(pattern={pattern!r}) — need >=2 comparable runs, skipping (returns NaN).")
        return float("nan")

    banks = []
    for p in paths:
        try:
            blob = torch.load(p, map_location="cpu")
            C = torch.as_tensor(blob["concepts"], dtype=torch.float32)
            C = _orient_atoms(C).cpu().numpy()
            banks.append((p, C))
        except Exception as e:
            print(f"[stability] skipping {p}: {e}")

    if len(banks) < 2:
        return float("nan")

    # Keep the largest group sharing the same feature dimension.
    by_d = {}
    for p, C in banks:
        by_d.setdefault(C.shape[1], []).append((p, C))
    d_key = max(by_d.keys(), key=lambda k: len(by_d[k]))
    banks = by_d[d_key]

    sims = []
    for i in range(len(banks)):
        for j in range(i + 1, len(banks)):
            sims.append(_mean_matched_cosine(banks[i][1], banks[j][1]))

    return float(np.mean(sims)) if sims else float("nan")


def print_atom_stats(C: torch.Tensor) -> None:
    """Print min, max, mean, and mean-absolute values per atom (row), plus global stats."""
    if C.ndim != 2:
        raise ValueError(f"Expected 2D matrix, got {tuple(C.shape)}")

    atom_max, atom_min = C.max(dim=1).values, C.min(dim=1).values
    atom_mean, atom_mean_abs = C.mean(dim=1), C.abs().mean(dim=1)

    print("=== Per-Atom Stats ===")
    for i, (mn, mx, mu, ma) in enumerate(zip(atom_min, atom_max, atom_mean, atom_mean_abs)):
        print(f"Atom {i:02d}: min = {mn.item():.6f}, max = {mx.item():.6f}, "
              f"mean = {mu.item():.6f}, mean|x| = {ma.item():.6f}")

    print("\n=== Global Stats ===")
    print(f"Global min      = {C.min().item():.6f}")
    print(f"Global max      = {C.max().item():.6f}")
    print(f"Global mean     = {C.mean().item():.6f}")
    print(f"Global mean|x|  = {C.abs().mean().item():.6f}")


# ---------- Inputs: fruits concept bank ----------
concept_file = "/media/NVME_8TB/abka03/Projects/xl-vlms/outputs/fruits/concept/snmf/combined_concept_snmf_cr0.5_raw.pth"
# Fruits currently has a single snmf run, so cross-run stability has no second run to
# compare against and will correctly report NaN below (see compute_cross_run_stability's
# printed message) rather than silently mixing in unrelated datasets/experiments.
stability_dir = "/media/NVME_8TB/abka03/Projects/xl-vlms/outputs/fruits"
stability_pattern = "**/concept/snmf/combined_concept_snmf_cr0.5_raw.pth"

# ---------- Load single concept file and compute atom metrics ----------
blob = torch.load(concept_file, map_location="cpu")
concepts = torch.as_tensor(blob["concepts"], dtype=torch.float32)
concepts = _orient_atoms(concepts)
print("concepts shape:", concepts.shape)

exact_zero_fraction = compute_exact_zero_fraction(concepts)
sparsity = compute_atom_sparsity(concepts)          # relative threshold, default 1%
selectivity = compute_atom_selectivity(concepts)     # Hoyer sparsity
overlap = compute_concept_overlap_tensor(concepts)   # normalized cosine similarity

# ---------- Cross-run stability ----------
stability = compute_cross_run_stability(stability_dir, stability_pattern)

print_atom_stats(concepts)

# ---------- Results ----------
results = {
    "exact_zero_fraction": exact_zero_fraction,     # literal % of entries == 0.0
    "sparsity_relative_1pct": sparsity,              # % of entries < 1% of that atom's max |x|
    "selectivity_hoyer": selectivity,                # 0..1, higher = more concentrated
    "overlap_cosine": overlap,                       # mean off-diagonal cosine sim, lower = more disentangled
    "stability_matched_cosine": stability,            # higher = more reproducible across runs; NaN if <2 comparable runs
}
print(results)
